In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import os

import numpy as np
import torch
from torch_geometric import seed_everything

seed_everything(42)

cache_dir = "./.cache_predql"

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: 

In [3]:
from experiments.model_training.utils import get_device

device = get_device()

Using device: cuda


In [4]:
import redelex

from relbench.datasets import get_dataset
from relbench.modeling.utils import get_stype_proposal
from predql_tasks.tasks import StatsUserBadgeTmpTask

In [5]:
dataset_name = "ctu-stats"
task_name = "stats_user_badge_tmp"

dataset = get_dataset("ctu-stats", download=False)
task = StatsUserBadgeTmpTask()
db = dataset.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.91 seconds.


In [6]:
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'postHistory': {'__PK__': <stype.numerical: 'numerical'>,
  'PostHistoryTypeId': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'RevisionGUID': <stype.text_embedded: 'text_embedded'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'Text': <stype.text_embedded: 'text_embedded'>,
  'Comment': <stype.text_embedded: 'text_embedded'>,
  'UserDisplayName': <stype.text_embedded: 'text_embedded'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'votes': {'__PK__': <stype.numerical: 'numerical'>,
  'PostId': <stype.numerical: 'numerical'>,
  'VoteTypeId': <stype.numerical: 'numerical'>,
  'CreationDate': <stype.timestamp: 'timestamp'>,
  'UserId': <stype.numerical: 'numerical'>,
  'BountyAmount': <stype.numerical: 'numerical'>,
  'FK_posts_PostId': <stype.numerical: 'numerical'>,
  'FK_users_UserId': <stype.numerical: 'numerical'>},
 'tags': {'__PK__': <st

In [7]:
from experiments.model_training.training.text_embedder import TextEmbedder 
from experiments.model_training.utils import patched_to_unix_time

import relbench.modeling.graph
import relbench.modeling.utils
from relbench.modeling.graph import make_pkey_fkey_graph
from torch_frame.config.text_embedder import TextEmbedderConfig


text_embedder = TextEmbedderConfig(
    text_embedder=TextEmbedder(
        "BAAI/bge-base-en-v1.5",
        device=device,
        cache_dir=cache_dir), batch_size=256
)

relbench.modeling.graph.to_unix_time = patched_to_unix_time
relbench.modeling.utils.to_unix_time = patched_to_unix_time

data, col_stats_dict = make_pkey_fkey_graph(
    db, 
    col_to_stype_dict, 
    text_embedder,
    cache_dir=os.path.join(cache_dir, dataset_name)
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/relbench/modeling/graph.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or m

In [8]:
data

HeteroData(
  postHistory={
    tf=TensorFrame([266825, 8]),
    time=[266825],
  },
  votes={
    tf=TensorFrame([295778, 5]),
    time=[295778],
  },
  tags={ tf=TensorFrame([1032, 4]) },
  posts={
    tf=TensorFrame([80899, 20]),
    time=[80899],
  },
  badges={
    tf=TensorFrame([69012, 3]),
    time=[69012],
  },
  users={
    tf=TensorFrame([34149, 13]),
    time=[34149],
  },
  postLinks={
    tf=TensorFrame([9580, 4]),
    time=[9580],
  },
  comments={
    tf=TensorFrame([154876, 6]),
    time=[154876],
  },
  (postHistory, f2p_FK_posts_PostId, posts)={ edge_index=[2, 266825] },
  (posts, rev_f2p_FK_posts_PostId, postHistory)={ edge_index=[2, 266825] },
  (postHistory, f2p_FK_users_UserId, users)={ edge_index=[2, 247052] },
  (users, rev_f2p_FK_users_UserId, postHistory)={ edge_index=[2, 247052] },
  (votes, f2p_FK_posts_PostId, posts)={ edge_index=[2, 295705] },
  (posts, rev_f2p_FK_posts_PostId, votes)={ edge_index=[2, 295705] },
  (votes, f2p_FK_users_UserId, users)={ edg

In [9]:
from experiments.model_training.utils import make_loaders

loader_dict = make_loaders(
    data=data,
    task=task,
    batch_size=256,
    num_neighbors=[128, 64, 32],
    is_temporal=True
)

Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.77 seconds.
Loading Database object from /home/kolesiko/.cache/relbench/ctu-stats/db...
Done in 0.75 seconds.


In [11]:
in_channels = 128
task_type = task.task_type
dropout = 0.4
is_temporal = True
epochs = 50

mlp_config = {
    "in_channels": 128,
    "hidden_channels": 64,
    "out_channels": 1,
    "layers": 2,
    "act": "relu",
    "norm": "batch_norm",
    "bias": True
}

## SAGE

In [16]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.sage_model import SAGEModel
import gc


sage_config = {
    "channels": 128,
    "aggr": "mean",
    "layers": 3,
}

sage_model = SAGEModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=sage_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(sage_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=sage_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="r2",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{sage_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del sage_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 541/541 [00:29<00:00, 18.64it/s]


Epoch 1/50, Train Loss: 0.0412


Evaluating: 100%|██████████| 113/113 [00:02<00:00, 48.94it/s]


Epoch 1/50, Eval Metrics: {'mae': 0.1460728794336319, 'mse': 0.07780484110116959, 'r2': -0.001779794692993164, 'loss': 0.0388509852107928}
New best model found with r2: -0.0018


Training: 100%|██████████| 541/541 [00:28<00:00, 19.03it/s]


...

Evaluating: 100%|██████████| 113/113 [00:02<00:00, 50.67it/s]


{'mae': 0.12226194888353348, 'mse': 0.06384742259979248, 'r2': 0.17792963981628418, 'loss': 0.031923710237257666}


Evaluating: 100%|██████████| 134/134 [00:02<00:00, 44.74it/s]


{'mae': 0.12143229693174362, 'mse': 0.06212860345840454, 'r2': 0.22816354036331177, 'loss': 0.03106430200279822}


0

## HGT

In [17]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.hgt_model import HGTModel
import gc


hgt_config = {
    "channels": 128,
    "heads": 4,
    "layers": 3
}

hgt_model = HGTModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=hgt_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(hgt_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=hgt_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="r2",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{hgt_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del hgt_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 541/541 [00:35<00:00, 15.31it/s]


Epoch 1/50, Train Loss: 0.0449


Evaluating: 100%|██████████| 113/113 [00:02<00:00, 49.21it/s]


Epoch 1/50, Eval Metrics: {'mae': 0.14609356224536896, 'mse': 0.06520048528909683, 'r2': 0.16050821542739868, 'loss': 0.03260022452519506}
New best model found with r2: 0.1605


Training: 100%|██████████| 541/541 [00:34<00:00, 15.58it/s]


...

Evaluating: 100%|██████████| 113/113 [00:02<00:00, 49.43it/s]


{'mae': 0.14588434994220734, 'mse': 0.06407484412193298, 'r2': 0.17500150203704834, 'loss': 0.03203742349130275}


Evaluating: 100%|██████████| 134/134 [00:02<00:00, 51.38it/s]


{'mae': 0.14243271946907043, 'mse': 0.06098227947950363, 'r2': 0.2424045205116272, 'loss': 0.030491139960875813}


0

## HAN

In [18]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.han_model import HANModel
import gc


han_config = {
    "channels": 128,
    "heads": 2,
    "dropout": 0.7,
    "layers": 3,
}

han_model = HANModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=han_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(han_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=han_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="r2",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{han_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del han_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 541/541 [00:31<00:00, 17.06it/s]


Epoch 1/50, Train Loss: 0.0483


Evaluating: 100%|██████████| 113/113 [00:02<00:00, 42.86it/s]


Epoch 1/50, Eval Metrics: {'mae': 0.26346656680107117, 'mse': 0.09462910890579224, 'r2': -0.21840131282806396, 'loss': 0.04731455141236868}
New best model found with r2: -0.2184


Training: 100%|██████████| 541/541 [00:31<00:00, 17.25it/s]


...

Evaluating: 100%|██████████| 113/113 [00:02<00:00, 37.95it/s]


{'mae': 0.16418489813804626, 'mse': 0.07778029143810272, 'r2': -0.0014636516571044922, 'loss': 0.03889014396920205}


Evaluating: 100%|██████████| 134/134 [00:03<00:00, 35.28it/s]


{'mae': 0.16695155203342438, 'mse': 0.08054694533348083, 'r2': -0.0006513595581054688, 'loss': 0.04027347292135483}


0

## GIN

In [19]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gin_model import GINModel
import gc


gin_config = {
    "channels": 128,
    "mlp_layers": 2,
    "mlp_dropout": 0.3,
    "mlp_act": "relu",
    "mlp_norm": "batch_norm",
    "mlp_bias": True,
    "train_eps": True,
    "layers": 3,
}

gin_model = GINModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gin_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(gin_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=gin_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="r2",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gin_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gin_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 541/541 [00:38<00:00, 13.93it/s]


Epoch 1/50, Train Loss: 0.0483


Evaluating: 100%|██████████| 113/113 [00:02<00:00, 40.39it/s]


Epoch 1/50, Eval Metrics: {'mae': 0.11896886676549911, 'mse': 0.06463845819234848, 'r2': 0.16774463653564453, 'loss': 0.03231922853689756}
New best model found with r2: 0.1677


Training: 100%|██████████| 541/541 [00:35<00:00, 15.19it/s]


...

Evaluating: 100%|██████████| 113/113 [00:02<00:00, 41.70it/s]


{'mae': 0.13681888580322266, 'mse': 0.06324242800474167, 'r2': 0.18571925163269043, 'loss': 0.03162121574745322}


Evaluating: 100%|██████████| 134/134 [00:03<00:00, 41.32it/s]


{'mae': 0.14027619361877441, 'mse': 0.0625026747584343, 'r2': 0.2235163450241089, 'loss': 0.03125133910795519}


0

## GATv2

In [20]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gatv2_model import GATv2Model
import gc


gatv2_config = {
    "channels": 128,
    "heads": 4,
    "add_self_loops": False,
    "layers": 3,
}

gatv2_model = GATv2Model(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gatv2_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.HuberLoss()
optimizer = torch.optim.AdamW(gatv2_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, verbose=True
)

trainer = Trainer(
    task=task,
    model=gatv2_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="r2",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gatv2_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gatv2_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Training: 100%|██████████| 541/541 [00:39<00:00, 13.81it/s]


Epoch 1/50, Train Loss: 0.0490


Evaluating: 100%|██████████| 113/113 [00:03<00:00, 36.67it/s]


Epoch 1/50, Eval Metrics: {'mae': 0.10630770772695541, 'mse': 0.08115333318710327, 'r2': -0.04489338397979736, 'loss': 0.04057666640905862}
New best model found with r2: -0.0449


Training: 100%|██████████| 541/541 [00:38<00:00, 13.93it/s]


...

Epoch 31/50, Train Loss: 0.0474


Evaluating: 100%|██████████| 113/113 [00:03<00:00, 35.71it/s]


Epoch 31/50, Eval Metrics: {'mae': 0.13976846635341644, 'mse': 0.07801806181669235, 'r2': -0.0045250654220581055, 'loss': 0.039009031772963956}


Training:  25%|██▌       | 136/541 [00:11<00:33, 12.11it/s]


KeyboardInterrupt: 